# CPython Set Internals, Hash Engines, and Interview Guide

## 1. THEORY LAYER
### Origin and Motivation
Sets were added to Python to provide mathematical set operations (union, intersection, etc.) with fast, constant-time checks for uniqueness.

### Memory Model
CPython implements sets as hash tables. Instead of key-value pairs (like dicts), sets store keys only.
- The set must support constant-time lookup $O(1)$.
- Uniqueness is enforced by calculating `hash(elem)` and resolving collisions via probing.

---

## 2. CPYTHON INTERNAL LAYER
### C struct Definition (`setobject.h`)
```c
typedef struct {
    PyObject_HEAD
    Py_ssize_t fill;         // Active + Dummy entries
    Py_ssize_t used;         // Active entries only
    Py_ssize_t mask;         // Size of table - 1
    setentry *table;         // Pointer to hash table entries
    setentry smalltable[8];  // Avoid malloc for small sets
} PySetObject;
```

### Collision Resolution and Resizing
- Python uses **open addressing** with perturbation probing.
- If the table is 2/3 full, it is resized to the next power of 2.

---


In [ ]:
import sys

print("=" * 70)
print("CPython Set Memory Tracking")
print("=" * 70)

s = set()
prev_size = sys.getsizeof(s)
for i in range(30):
    s.add(i)
    curr_size = sys.getsizeof(s)
    if curr_size != prev_size:
        print(f"Set size: {len(s):>2} | Memory: {curr_size:>3} bytes | Resize Triggered!")
        prev_size = curr_size



---
## 3. COMPLETE METHOD REFERENCE

### 3.1 `add(elem)`
- **Syntax**: `set.add(elem)`
- **Time Complexity**: $O(1)$ average.


In [ ]:
print("\n=== Method: add ===")
s = {1, 2}
s.add(3)
print(f"add(3) -> {s}")



### 3.2 `remove(elem)` vs `discard(elem)`
- **Syntax**: `set.remove(elem)` / `set.discard(elem)`
- **Behavior**: `remove()` raises `KeyError` if element is missing; `discard()` remains silent.


In [ ]:
print("\n=== Methods: remove and discard ===")
s = {1, 2, 3}
s.discard(99) # Safe
try:
    s.remove(99)
except KeyError as e:
    print(f"remove(99) raised KeyError: {e}")



### 3.3 `pop()`
- **Syntax**: `set.pop()`
- **Time Complexity**: $O(1)$.
- **Behavior**: Removes and returns an arbitrary element.


In [ ]:
print("\n=== Method: pop ===")
s = {10, 20, 30}
val = s.pop()
print(f"pop() -> {val}, remaining set: {s}")



### 3.4 `union(*others)`
- **Syntax**: `set.union(*others)` or `|` operator.
- **Time Complexity**: $O(n + m)$.


In [ ]:
print("\n=== Method: union ===")
s1 = {1, 2}
s2 = {3, 4}
print(f"union -> {s1.union(s2)}")



### 3.5 `intersection(*others)`
- **Syntax**: `set.intersection(*others)` or `&` operator.
- **Time Complexity**: $O(\min(n, m))$.


In [ ]:
print("\n=== Method: intersection ===")
s1 = {1, 2, 3}
s2 = {2, 3, 4}
print(f"intersection -> {s1.intersection(s2)}")



---
## 4. IMPLEMENTATION LAYER
### Level 1: Pure Python Hash Set Reimplementation


In [ ]:
class CustomHashSet:
    """Manual reimplementation of a Hash Set in Pure Python."""
    def __init__(self, size=8):
        self.size = size
        self.slots = [None] * self.size
        self.count = 0

    def _hash(self, key):
        return hash(key) % self.size

    def add(self, key):
        if self.count / self.size > 0.66:
            self._resize()
        
        idx = self._hash(key)
        # Linear probing for slot collision resolution
        while self.slots[idx] is not None:
            if self.slots[idx] == key:
                return  # Duplicate found
            idx = (idx + 1) % self.size
            
        self.slots[idx] = key
        self.count += 1

    def __contains__(self, key):
        idx = self._hash(key)
        start_idx = idx
        while self.slots[idx] is not None:
            if self.slots[idx] == key:
                return True
            idx = (idx + 1) % self.size
            if idx == start_idx:
                break
        return False

    def _resize(self):
        old_slots = self.slots
        self.size *= 2
        self.slots = [None] * self.size
        self.count = 0
        for item in old_slots:
            if item is not None:
                self.add(item)

    def __repr__(self):
        return "{" + ", ".join(str(x) for x in self.slots if x is not None) + "}"

print("\n=== Level 1: Custom Hash Set ===")
h_set = CustomHashSet()
for num in [1, 5, 9, 13]:  # Generate potential modulo overlaps
    h_set.add(num)
print(f"Custom set: {h_set} | Contains 5: {5 in h_set}")



### Level 2: Python Built-in Comparison
Verify behavior, operators, and benchmarks.


In [ ]:
print("\n=== Level 2: Set Operators ===")
A = {1, 2, 3}
B = {3, 4, 5}
print(f"Difference (A - B): {A - B}")
print(f"Symmetric Difference (A ^ B): {A ^ B}")



### Level 3: Advanced Usage Systems — Deduplication Engine


In [ ]:
class DeduplicationEngine:
    """Deduplicates massive logs using set hashes for fast memory checks."""
    def __init__(self):
        self.seen = set()

    def process_log(self, log_id):
        if log_id in self.seen:
            return False  # Already processed
        self.seen.add(log_id)
        return True

engine = DeduplicationEngine()
print(f"Process unique log: {engine.process_log('LOG_001')}")
print(f"Process duplicate log: {engine.process_log('LOG_001')}")



---
## 5. EXPERIMENTATION LAYER
### Set vs List Membership Performance Testing


In [ ]:
print("\n=== Section 5: Set vs List Performance ===")
import time

items = list(range(10000))
items_set = set(items)

# List search
t0 = time.perf_counter()
for _ in range(1000):
    _ = 9999 in items
t_list = (time.perf_counter() - t0) * 1000

# Set search
t0 = time.perf_counter()
for _ in range(1000):
    _ = 9999 in items_set
t_set = (time.perf_counter() - t0) * 1000

print(f"List search time: {t_list:.3f} ms")
print(f"Set search time:  {t_set:.3f} ms")
print(f"Speedup factor:   {t_list / (t_set + 1e-10):.1f}x")



---
## 6. VISUALIZATION LAYER
```
CPython Set Table (Open Addressing + Probing):
Hash Slot  ->   Key Value
[ 0 ]      ->   None
[ 1 ]      ->   15 (hash match)
[ 2 ]      ->   22 (collision fallback)
[ 3 ]      ->   None
```
---
## 7. INTERVIEW MASTER LAYER

### 50 Scenario-Based Interview Q&As (Summary Selection)

1. **How do sets guarantee uniqueness?**
   - Before inserting, Python checks the slot derived from the element's hash. If occupied, value equality (`__eq__`) is evaluated to determine if it is a duplicate.
2. **What is the difference between set and frozenset?**
   - `set` is mutable (not hashable). `frozenset` is immutable and hashable (can be used as dictionary keys).
3. **What is the default load factor for resizing in set?**
   - Resizing triggers when table load reaches 2/3 capacity (`fill * 3 >= (mask + 1) * 2`).
